* **概要**: 1990年代のインターネット掲示板（ニュースグループ）の投稿を集めたデータセットです。
* **データ数**: 約18,000件のテキスト文書。
これに対して、LDAモデルを適用してみる

In [55]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score

異なるカテゴリーの記事を4つ,そして関係ないものを取り除く
datesetの中身(Bunch)は、dataset.data:テキストの集合,dataset.target:カテゴリの数値ラベル,dataset.target_names:カテゴリの名前リスト見たいな感じ

In [56]:
categories = ['alt.atheism', 'comp.graphics', 'sci.space', 'rec.sport.baseball']

# データの取得（ヘッダーやフッターなどのノイズを除去して純粋なテキストのみを取得）
print("Loading dataset...")
dataset = fetch_20newsgroups(subset='train', categories=categories, 
                             shuffle=True, random_state=42,
                             remove=('headers', 'footers', 'quotes'))
raw_texts = dataset.data
print(f"Loaded {len(raw_texts)} documents.")

Loading dataset...
Loaded 2254 documents.


In [57]:
raw_texts[0:3]

["\nA 68070 is just a 68010 with a built in MMU.  I don't even think that Moto.\nmanufactures them.\n\n                                  - Ian Romanick\n                                    Dancing Fool of Epsilon",
 "Hello, I realize that this might be a FAQ but I have to ask since I don't get a\nchange to read this newsgroup very often.  Anyways for my senior project I need\nto convert an AutoCad file to a TIFF file.  Please I don't need anyone telling\nme that the AutoCAD file is a vector file and the TIFF is a bit map since I\nhave heard that about 100 times already I would just like to know if anyone\nknows how to do this or at least point me to the right direction.",
 '\n\n\nHow do you know it\'s based on ignorance, couldn\'t that be wrong? Why would it\nbe wrong \nto fall into the trap that you mentioned? \n\nAlso, if I may, what the heck where we talking about and why didn\'t I keep \nsome comments on there to see what the line of thoughts were?\n\nMAC\n \n\n\n\n\n\n--\n********

countvectorizerで、
95%以上の文章に存在する単語は無視する、２個未満の単語は無視する。頻出頻度の高い単語(1000)に絞る,意味のない単語は消す
Xは、疎行列.もし普通に行列を作ると(d*V)の数になるので、メモリが破綻する。そのため、0の記録はしない。1の記録のみする
vocabは辞書こと。numpyの配列で渡されている

In [58]:
vectorizer = CountVectorizer(max_df=0.95, min_df=2, 
                             max_features=1000, 
                             stop_words='english')


X = vectorizer.fit_transform(raw_texts)

# 辞書（IDから実際の単語文字列へのマッピング）を取得
vocab = vectorizer.get_feature_names_out()

In [59]:
print(type(X))

<class 'scipy.sparse._csr.csr_matrix'>


ここからLDAの具体的な内容について設定
トピック数は、4で決定
推定法はSVIを利用している。また、繰り返し学習数も10(ここでの10は全データを10回使うの意味、batch_size=128)にしている
推定結果の確かめ方として、文章がどのトピックなのか、確率が高いところを採用して、それを予測値とする
それの性能評価として、正答率は,ARIで判断する(ランダムに記事を２個選んで、その２つが正しい別れ方をしていたら1点のようにする,まぐれで当たる分もあるのでそこはマイナスにしてあるのようなもの)
その値の平均値(乱数で変化)をモデルの評価とする。

In [60]:
n_topics = 4
result=[]
for i in range(50):
   lda = LatentDirichletAllocation(n_components=n_topics, 
                                max_iter=10, 
                                learning_method='online', 
                                random_state=i)
    #推定の実行（ここで裏側の期待値計算と足し算が高速で行われています）
   lda.fit(X)
   topic_distribution = lda.transform(X)
   predicted_topics = np.argmax(topic_distribution, axis=1)
   score = adjusted_rand_score(dataset.target, predicted_topics)
   result.append(score)
print(np.mean(result))

0.29879110262267766


※補足:そのトピックにおける上位の単語の出し方

model.components_ は (K次元, V次元) の 行列のこと。
topic.argsort()で、行列の値を小さい順に並べて、-10個、つまり大きい順に10個取り出して、インデックス番号を取り出す

In [61]:
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic #{topic_idx + 1}:")
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        print(" ".join(top_features))

print_top_words(lda, vocab, n_top_words=10)


Topic #1:
god year good don just think time like believe game

Topic #2:
image edu graphics software data file available images ftp files

Topic #3:
don just think people like know does point problem use

Topic #4:
space nasa launch earth orbit satellite shuttle lunar data moon
